# RiskModels — funds, benchmarks & 13F (HTTP golden path)

**[Get API key](https://riskmodels.app/get-key)** · **[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/surface_funds_http.ipynb)**

The Python SDK **does not** wrap funds endpoints yet (as of `riskmodels-py` 0.3.x). This notebook uses the same **authenticated `requests` session** as the other quickstarts (`quickstart_connect()`).

**What you will hit:** fund **search**, a **benchmark** surface (**SPY**), and **13F filer** discovery — three different `source_kind` surfaces on the data plane.


### Colab only

Install **`requests`** + **`python-dotenv`** + **`riskmodels-py`** (only for `riskmodels.notebook` key helpers).


In [ ]:
import sys

try:
    import google.colab  # noqa: F401
    _COLAB = True
except ImportError:
    _COLAB = False

if _COLAB:
    import subprocess

    _deps = ["requests", "python-dotenv"]
    _pypi = "riskmodels-py>=0.3.4"
    _git = (
        "riskmodels-py @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py from PyPI (+ requests, python-dotenv).")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print(
            "Colab: installed riskmodels-py from GitHub main "
            "(PyPI may not list this version yet)."
        )
else:
    print("Local: pip install riskmodels-py requests python-dotenv")


## 1. Session — `quickstart_connect()`

Pings **`/balance`** so you fail fast if the key is missing or invalid.


In [ ]:
from riskmodels.notebook import quickstart_connect

session, BASE_URL, _key = quickstart_connect()
print("BASE_URL =", BASE_URL)


## 2. Mutual funds / ETFs — `GET /data/funds/search`

Resolve a **`bw_fund_id`** from a human query (mutual fund name or ETF ticker).


In [ ]:
params = {"q": "ARKK"}
r = session.get(f"{BASE_URL}/data/funds/search", params=params, timeout=60)
r.raise_for_status()
body = r.json()
results = body.get("results") or body.get("funds") or body.get("data") or []
print("keys:", list(body.keys())[:12])
if results:
    first = results[0]
    print("first hit:", {k: first.get(k) for k in list(first)[:8]})
else:
    print("(no results — try another query)")


## 3. Benchmark portfolio — `GET /data/benchmark/{id}`

Alias **`SPY`** resolves to a **BenchmarkContext** + constituents (reference sleeve, not a fund).


In [ ]:
alias = "SPY"
r = session.get(f"{BASE_URL}/data/benchmark/{alias}", params={"top": 15}, timeout=60)
r.raise_for_status()
bench = r.json()
print("top-level keys:", list(bench.keys())[:14])
print("benchmark_kind:", bench.get("benchmark_kind"), "| name:", bench.get("name"))


## 4. 13F filers — `GET /13f/filers/search`

Institutional **quarterly** filers — **no NAV**. Follow-up calls use **`bw_filer_id`** from search.


In [ ]:
r = session.get(f"{BASE_URL}/13f/filers/search", params={"q": "Berkshire", "limit": 5}, timeout=60)
r.raise_for_status()
fbody = r.json()
rows = fbody.get("results") or []
print("n_results:", len(rows))
if rows:
    print("first:", {k: rows[0].get(k) for k in list(rows[0])[:10]})


## Next steps

- **Stocks (SDK):** [`surface_stocks_sdk.ipynb`](./surface_stocks_sdk.ipynb)
- **Full ladder (REST + SDK + AOM):** [`riskmodels_quickstart.ipynb`](./riskmodels_quickstart.ipynb)
- **OpenAPI** (all paths / schemas): repo root `OPENAPI_SPEC.yaml`.
